# 03. SOLAR-10.7B-Instruct 학습 & 추론

**upstage/SOLAR-10.7B-Instruct-v1.0** 모델을 QLoRA로 파인튜닝하고, dev 평가 및 test 제출 파일을 생성합니다.

| 항목 | 값 |
|---|---|
| 베이스 모델 | `upstage/SOLAR-10.7B-Instruct-v1.0` |
| 파인튜닝 방식 | QLoRA (4-bit NF4 + LoRA) |
| 추론 가속 | `unsloth FastLanguageModel` |
| 학습 프레임워크 | `trl SFTTrainer` |

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env')

# ── 실험 설정 (여기만 수정) ────────────────────────────────────────────
BASE_MODEL     = "upstage/SOLAR-10.7B-Instruct-v1.0"
RUN_NAME       = "solar-r32-lr1e4-ep3-v2"

# LoRA
LORA_R         = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.05

# 학습
NUM_EPOCHS     = 3
LEARNING_RATE  = 1e-4
BATCH_SIZE     = 1
GRAD_ACCUM     = 16      # 실효 배치 = BATCH_SIZE × GRAD_ACCUM
MAX_SEQ_LEN    = 2048
MAX_NEW_TOKENS = 256
SAVE_STEPS     = 200

# 경로
TRAIN_PATH     = "../data/raw/train.csv"
DEV_PATH       = "../data/raw/dev.csv"
TEST_PATH      = "../data/raw/test.csv"
OUTPUT_DIR     = "../outputs/checkpoints"
PRED_DIR       = "../outputs/predictions"
# ──────────────────────────────────────────────────────────────────────

print(f"모델    : {BASE_MODEL}")
print(f"Run Name: {RUN_NAME}")
print(f"실효배치: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. 데이터 로드 및 전처리

In [ ]:
from src.data.preprocessor import DialoguePreprocessor

preprocessor = DialoguePreprocessor()
train_df = preprocessor.process(pd.read_csv(TRAIN_PATH))
dev_df   = preprocessor.process(pd.read_csv(DEV_PATH))
test_df  = preprocessor.process(pd.read_csv(TEST_PATH))

print(f"train: {len(train_df):,}  |  dev: {len(dev_df):,}  |  test: {len(test_df):,}")
train_df.head(2)

## 2. 프롬프트 템플릿

In [ ]:
PROMPT = """### System:
당신은 한국어 대화 요약 전문가입니다. 대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. 요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. 핵심 내용만 1~3문장으로 간결하게 요약하세요. 반드시 한국어로 작성하되, CPU·USB 등 고유 영문 약어나 전문 용어는 그대로 사용해도 됩니다.

### User:
아래 대화를 읽고 핵심 내용을 한국어로 요약해주세요. 화자 태그(#Person1# 등)를 유지하세요.

{dialogue}

### Assistant:
{summary}"""

# 샘플 확인
print(PROMPT.format(dialogue=train_df['dialogue'].iloc[0][:200], summary=train_df['summary'].iloc[0]))

## 3. 모델 로드 (4-bit QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"[1/4] 모델 로드: {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

total = sum(p.numel() for p in model.parameters())
print(f"[1/4] 로드 완료 | 총 파라미터: {total:,}")

## 4. LoRA 적용

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"[2/4] LoRA 적용 (r={LORA_R}, alpha={LORA_ALPHA})")

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"[2/4] 학습 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. 데이터셋 구성

In [ ]:
from datasets import Dataset

print(f"[3/4] 데이터셋 구성: {len(train_df):,}개")

records = [
    {"text": PROMPT.format(dialogue=row["dialogue"], summary=row["summary"])}
    for _, row in train_df.iterrows()
]
train_dataset = Dataset.from_list(records)

print(f"샘플 텍스트 길이: {len(train_dataset[0]['text'])}자")
print(train_dataset[0]['text'][:300])

## 6. 학습

In [ ]:
from datetime import datetime
from trl import SFTTrainer, SFTConfig

EXP_ID     = datetime.now().strftime("EXP-%Y%m%d-%H%M%S")
output_dir = str(Path(OUTPUT_DIR) / EXP_ID)
use_bf16   = torch.cuda.is_bf16_supported()

print(f"[4/4] 학습 시작")
print(f"  실험 ID  : {EXP_ID}")
print(f"  저장 경로: {output_dir}")

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=use_bf16,
    fp16=not use_bf16,
    seed=42,
    logging_steps=10,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()
print("[4/4] 학습 완료")

## 7. 체크포인트 저장

In [ ]:
ckpt_path = str(Path(OUTPUT_DIR) / EXP_ID / "final")
model.save_pretrained(ckpt_path)
tokenizer.save_pretrained(ckpt_path)
print(f"체크포인트 저장 완료: {ckpt_path}")

## 8. Dev 평가 (ROUGE)

> 학습된 모델로 dev 세트를 추론하고 ROUGE 점수를 계산합니다.

In [ ]:
from tqdm import tqdm
from src.utils.rouge_evaluator import RougeEvaluator

# bf16 학습 후 추론 시 dtype 맞춤
if use_bf16:
    model = model.to(torch.bfloat16)
model.eval()

def generate(dialogue: str) -> str:
    prompt = PROMPT.format(dialogue=dialogue, summary="")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

EVAL_SAMPLES = 499
sample_df = dev_df.sample(n=min(EVAL_SAMPLES, len(dev_df)), random_state=42)
print(f"Dev 평가 ({len(sample_df)}/{len(dev_df)}개)")

preds, refs = [], []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="eval"):
    preds.append(generate(row["dialogue"]))
    refs.append(row["summary"])

scores = RougeEvaluator().score(preds, refs)
print(f"\n  ROUGE-1 : {scores['rouge1']:.4f}")
print(f"  ROUGE-2 : {scores['rouge2']:.4f}")
print(f"  ROUGE-L : {scores['rougeL']:.4f}")
print(f"  Score   : {scores['score']:.4f}")

## 9. Test 추론 및 제출 파일 생성

In [ ]:
print(f"Test 추론 ({len(test_df)}개)")

predictions = []
for dialogue in tqdm(test_df["dialogue"], desc="test"):
    predictions.append(generate(dialogue))

out_dir  = Path(PRED_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{RUN_NAME}.csv"

result_df = pd.DataFrame({"fname": test_df["fname"], "summary": predictions})
result_df.to_csv(out_path, index=False)
print(f"저장 완료 → {out_path}  ({len(result_df)}행)")
result_df.head()

---
## (선택) 기존 체크포인트로 추론만 실행

학습 없이 저장된 체크포인트로 바로 추론할 때 사용합니다.

In [ ]:
# ── 기존 체크포인트 경로 지정 ──────────────────────────────────────────
LOAD_CHECKPOINT = "../outputs/checkpoints/EXP-20260328-125900/checkpoint-2337"
# ──────────────────────────────────────────────────────────────────────

from unsloth import FastLanguageModel
from peft import PeftModel

print(f"베이스 모델 로드: {BASE_MODEL}")
inf_model, inf_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

print(f"LoRA 어댑터 로드: {LOAD_CHECKPOINT}")
inf_model = PeftModel.from_pretrained(inf_model, LOAD_CHECKPOINT)
FastLanguageModel.for_inference(inf_model)
inf_model.eval()
print("로드 완료")

def generate_from_ckpt(dialogue: str) -> str:
    prompt = PROMPT.format(dialogue=dialogue, summary="")
    inputs = inf_tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=MAX_SEQ_LEN).to("cuda")
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=inf_tokenizer.eos_token_id,
        )
    return inf_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

# 단일 샘플 테스트
print(generate_from_ckpt(test_df['dialogue'].iloc[0]))